DESCRIPTIVE PROFILING

This script takes the cleaned dataset produced in Phase 1
(cleaned_dataset.csv) and answers the question:

    "Who are the digitally excluded individuals?"

For each socio-demographic dimension (gender, age, education, geographic
area, type of municipality, employment status, household income), it
computes the weighted distribution of the digital exclusion status
(No internet access / Internet, low digital use / Internet, digital
adopter) and produces a stacked bar chart.

It also produces a summary table with the weighted "digital exclusion rate"
(share of "No internet access" + "Internet, low digital use") by group,
and a chart of the weighted mean financial literacy score by digital
exclusion status and by education level.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN_PATH = "cleaned_dataset.csv"

df = pd.read_csv(IN_PATH)
print(f"Loaded cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")


Loaded cleaned dataset: 4862 rows, 16 columns


1. COLLAPSED CATEGORIES FOR READABILITY

In [ ]:
# Age: merge the small 18-19 bracket into 20-29 -> "18-29"
age_map = {
    "18-19": "18-29", "20-29": "18-29",
    "30-39": "30-39", "40-49": "40-49", "50-59": "50-59",
    "60-69": "60-69", "70-79": "70-79",
}
df["age_group"] = df["age_bracket"].map(age_map)
AGE_ORDER = ["18-29", "30-39", "40-49", "50-59", "60-69", "70-79"]

# Employment: collapse into 4 broad groups
employment_map = {
    "Self-employed": "Employed",
    "Employee": "Employed",
    "Employee (temporary)": "Employed",
    "Retired": "Retired",
    "Not employed - looking for work": "Not in labour force",
    "Not employed - not looking for work": "Not in labour force",
    "Unable to work": "Not in labour force",
    "Homemaker": "Not in labour force",
    "Student": "Student",
}
df["employment_group"] = df["employment"].map(employment_map)
EMPLOYMENT_ORDER = ["Employed", "Retired", "Not in labour force", "Student"]

# Type of municipality: collapse into Rural / Town / Urban
municipality_map = {
    "Up to 2,000 inhabitants": "Rural (<10k)",
    "2,001-10,000 inhabitants": "Rural (<10k)",
    "10,001-50,000 inhabitants": "Town (10k-50k)",
    "50,001-250,000 inhabitants": "Urban (>50k)",
    "More than 250,000 inhabitants": "Urban (>50k)",
}
df["area_type"] = df["municipality_size"].map(municipality_map)
AREA_ORDER = ["Rural (<10k)", "Town (10k-50k)", "Urban (>50k)"]

REGION_ORDER = ["North-West", "North-East", "Centre", "South", "Islands"]
EDUCATION_ORDER = ["Low", "Medium", "High"]
INCOME_ORDER = ["Up to 1,750 EUR", "1,751-2,900 EUR", "Over 2,900 EUR", "Not declared"]
GENDER_ORDER = ["Male", "Female"]

EXCLUSION_ORDER = ["No internet access", "Internet, low digital use", "Internet, digital adopter"]
EXCLUSION_COLORS = {
    "No internet access": "#B0413E",
    "Internet, low digital use": "#E8A33D",
    "Internet, digital adopter": "#4C7C7C",
}

2. HELPER: WEIGHTED CROSS-TAB (ROW PERCENTAGES)

In [ ]:
def weighted_crosstab(data, group_var, order=None):
    """Return a DataFrame of weighted row percentages of
    digital_exclusion_status within each category of group_var."""
    sub = data.dropna(subset=[group_var, "digital_exclusion_status"])
    tab = sub.groupby([group_var, "digital_exclusion_status"])["weight"].sum().unstack(fill_value=0)
    tab = tab.reindex(columns=EXCLUSION_ORDER, fill_value=0)
    pct = tab.div(tab.sum(axis=1), axis=0) * 100
    if order is not None:
        pct = pct.reindex(order)
    return pct


def group_size_pct(data, group_var, order=None):
    """Weighted share of the (valid) sample in each category of group_var."""
    sub = data.dropna(subset=[group_var, "digital_exclusion_status"])
    sizes = sub.groupby(group_var)["weight"].sum()
    pct = sizes / sizes.sum() * 100
    if order is not None:
        pct = pct.reindex(order)
    return pct

3. PLOTTING HELPER

In [ ]:
def plot_stacked_bar(pct_table, title, filename, group_sizes=None):
    fig, ax = plt.subplots(figsize=(8, max(2.8, 0.7 * len(pct_table))))
    left = np.zeros(len(pct_table))
    categories = pct_table.index.astype(str)

    for status in EXCLUSION_ORDER:
        values = pct_table[status].values
        ax.barh(categories, values, left=left, label=status,
                color=EXCLUSION_COLORS[status], edgecolor="white", height=0.6)
        left += values

    # annotate with sample composition (% of sample) if provided
    if group_sizes is not None:
        ylabels = [f"{c}  (n={group_sizes.loc[c]:.1f}%)" for c in pct_table.index]
        ax.set_yticks(range(len(pct_table)))
        ax.set_yticklabels(ylabels)

    # keep the same top-to-bottom order as the input table
    ax.invert_yaxis()

    ax.set_xlim(0, 100)
    ax.set_xlabel("Weighted share (%)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {filename}")

4. OVERALL DISTRIBUTION (BASELINE)

In [ ]:
overall = df.dropna(subset=["digital_exclusion_status"])
overall_pct = (overall.groupby("digital_exclusion_status")["weight"].sum()
                / overall["weight"].sum() * 100).reindex(EXCLUSION_ORDER)

fig, ax = plt.subplots(figsize=(6, 2.2))
left = 0
for status in EXCLUSION_ORDER:
    val = overall_pct[status]
    ax.barh(["Full sample"], [val], left=left, color=EXCLUSION_COLORS[status],
            edgecolor="white", label=status)
    ax.text(left + val / 2, 0, f"{val:.1f}%", ha="center", va="center",
            color="white", fontweight="bold", fontsize=10)
    left += val
ax.set_xlim(0, 100)
ax.set_xlabel("")
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.set_title("Overall digital exclusion status (weighted %)", fontsize=12, fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("fig_overall_distribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved fig_overall_distribution.png")

Saved fig_overall_distribution.png


5. CROSS-TABS BY SOCIO-DEMOGRAPHIC VARIABLE

In [ ]:
breakdowns = {
    "gender": ("Gender", GENDER_ORDER),
    "age_group": ("Age group", AGE_ORDER),
    "education": ("Education level", EDUCATION_ORDER),
    "region": ("Geographic area", REGION_ORDER),
    "area_type": ("Type of municipality", AREA_ORDER),
    "employment_group": ("Employment status", EMPLOYMENT_ORDER),
    "income_bracket": ("Household income", INCOME_ORDER),
}

all_tables = []
exclusion_rate_rows = []

for var, (label, order) in breakdowns.items():
    pct = weighted_crosstab(df, var, order=order)
    sizes = group_size_pct(df, var, order=order)

    plot_stacked_bar(pct, f"Digital exclusion status by {label} (weighted %)",
                      f"fig_exclusion_by_{var}.png", group_sizes=sizes)

    # long format for the summary table
    long = pct.reset_index().melt(id_vars=var, var_name="digital_exclusion_status",
                                    value_name="weighted_pct")
    long["breakdown_variable"] = label
    long = long.rename(columns={var: "category"})
    all_tables.append(long[["breakdown_variable", "category", "digital_exclusion_status", "weighted_pct"]])

    # headline exclusion rate = "No internet" + "low digital use"
    excl_rate = pct["No internet access"] + pct["Internet, low digital use"]
    for cat, val in excl_rate.items():
        exclusion_rate_rows.append({
            "breakdown_variable": label,
            "category": cat,
            "digital_exclusion_rate_pct": round(val, 1),
            "sample_share_pct": round(sizes.loc[cat], 1),
        })

summary_tables = pd.concat(all_tables, ignore_index=True)
summary_tables["weighted_pct"] = summary_tables["weighted_pct"].round(1)
summary_tables.to_csv("summary_tables.csv", index=False)
print("Saved summary_tables.csv")

exclusion_rate_df = pd.DataFrame(exclusion_rate_rows)
exclusion_rate_df.to_csv("exclusion_rate_by_group.csv", index=False)
print("Saved exclusion_rate_by_group.csv")

# Overall exclusion rate, for reference
overall_excl_rate = overall_pct["No internet access"] + overall_pct["Internet, low digital use"]
print(f"\nOverall weighted digital exclusion rate: {overall_excl_rate:.1f}%")

Saved fig_exclusion_by_gender.png
Saved fig_exclusion_by_age_group.png
Saved fig_exclusion_by_education.png
Saved fig_exclusion_by_region.png
Saved fig_exclusion_by_area_type.png
Saved fig_exclusion_by_employment_group.png
Saved fig_exclusion_by_income_bracket.png
Saved summary_tables.csv
Saved exclusion_rate_by_group.csv

Overall weighted digital exclusion rate: 43.0%


6. FINANCIAL LITERACY SCORE BY EXCLUSION STATUS AND EDUCATION

In [ ]:
lit = df.dropna(subset=["digital_exclusion_status", "education"])

def weighted_mean(group):
    return np.average(group["financial_literacy_score"], weights=group["weight"])

lit_table = (lit.groupby(["education", "digital_exclusion_status"])
             .apply(weighted_mean, include_groups=False)
             .unstack())
lit_table = lit_table.reindex(index=EDUCATION_ORDER, columns=EXCLUSION_ORDER)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(EDUCATION_ORDER))
width = 0.25
for i, status in enumerate(EXCLUSION_ORDER):
    ax.bar(x + (i - 1) * width, lit_table[status].values, width,
           label=status, color=EXCLUSION_COLORS[status])

ax.set_xticks(x)
ax.set_xticklabels(EDUCATION_ORDER)
ax.set_ylabel("Weighted mean score (0-7)")
ax.set_title("Financial literacy score by education level\nand digital exclusion status",
              fontsize=12, fontweight="bold")
ax.set_ylim(0, 7)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("fig_literacy_by_exclusion_education.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved fig_literacy_by_exclusion_education.png")

print("\n--- Financial literacy score (weighted mean) by education x exclusion status ---")
print(lit_table.round(2))

Saved fig_literacy_by_exclusion_education.png

--- Financial literacy score (weighted mean) by education x exclusion status ---
digital_exclusion_status  No internet access  Internet, low digital use  \
education                                                                 
Low                                     3.38                       4.48   
Medium                                  2.92                       3.44   
High                                    2.23                       3.03   

digital_exclusion_status  Internet, digital adopter  
education                                            
Low                                            4.54  
Medium                                         4.06  
High                                           3.35  


7. KEY NUMBERS PRINTED FOR THE REPORT

In [ ]:
print("\n--- Headline digital exclusion rates by group (selected) ---")
for var, (label, order) in breakdowns.items():
    sub = exclusion_rate_df[exclusion_rate_df["breakdown_variable"] == label]
    print(f"\n{label}:")
    for _, row in sub.iterrows():
        print(f"  {row['category']}: {row['digital_exclusion_rate_pct']}% "
              f"(sample share {row['sample_share_pct']}%)")


--- Headline digital exclusion rates by group (selected) ---

Gender:
  Male: 48.8% (sample share 50.8%)
  Female: 36.9% (sample share 49.2%)

Age group:
  18-29: 53.9% (sample share 13.5%)
  30-39: 23.9% (sample share 18.3%)
  40-49: 23.5% (sample share 18.1%)
  50-59: 34.0% (sample share 20.3%)
  60-69: 60.1% (sample share 15.8%)
  70-79: 76.1% (sample share 14.0%)

Education level:
  Low: 18.0% (sample share 9.8%)
  Medium: 33.9% (sample share 65.0%)
  High: 75.9% (sample share 25.2%)

Geographic area:
  North-West: 40.8% (sample share 26.7%)
  North-East: 44.0% (sample share 19.7%)
  Centre: 39.1% (sample share 19.9%)
  South: 47.5% (sample share 22.7%)
  Islands: 43.9% (sample share 11.0%)

Type of municipality:
  Rural (<10k): 44.6% (sample share 41.0%)
  Town (10k-50k): 43.7% (sample share 35.8%)
  Urban (>50k): 38.9% (sample share 23.2%)

Employment status:
  Employed: 23.7% (sample share 57.6%)
  Retired: 71.7% (sample share 21.5%)
  Not in labour force: 66.2% (sample share 1

8. DIAGNOSTIC: AGE COMPOSITION WITHIN EDUCATION LEVELS

In [ ]:
# The "High education" group shows the highest digital exclusion rate
# (75.9%), which seems counterintuitive. As discussed in the report (10b),
# this is largely an age/cohort effect: in this sample "High education" is
# disproportionately composed of older respondents (university attendance
# was much less common decades ago), while "Low education" is concentrated
# among middle-aged respondents. This chart documents that composition.

age_edu = df.dropna(subset=["education", "age_group"])
age_edu_tab = (age_edu.groupby(["education", "age_group"])["weight"].sum()
               .unstack(fill_value=0))
age_edu_tab = age_edu_tab.div(age_edu_tab.sum(axis=1), axis=0) * 100
age_edu_tab = age_edu_tab.reindex(index=EDUCATION_ORDER, columns=AGE_ORDER)

fig, ax = plt.subplots(figsize=(8, 3.2))
left = np.zeros(len(age_edu_tab))
age_colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(AGE_ORDER)))
for i, age_grp in enumerate(AGE_ORDER):
    values = age_edu_tab[age_grp].values
    ax.barh(age_edu_tab.index.astype(str), values, left=left, label=age_grp,
            color=age_colors[i], edgecolor="white", height=0.6)
    left += values

ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Weighted share (%)")
ax.set_title("Age composition within each education level (weighted %)",
              fontsize=12, fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=6, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("fig_age_composition_by_education.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved fig_age_composition_by_education.png")
print("\n--- Age composition (%) within education level ---")
print(age_edu_tab.round(1))

Saved fig_age_composition_by_education.png

--- Age composition (%) within education level ---
age_group  18-29  30-39  40-49  50-59  60-69  70-79
education                                          
Low          7.2   27.3   28.0   25.5    7.6    4.4
Medium      16.7   21.8   19.4   20.7   13.7    7.6
High         8.8    6.2   10.3   17.0   24.3   33.5
